In [ ]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, Document, VectorStoreIndex, Settings, PromptTemplate, StorageContext, KnowledgeGraphIndex
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine, KnowledgeGraphQueryEngine
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Context, Event
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_parse import LlamaParse

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, util

import pandas as pd, re, ast, textwrap
from sqlalchemy import create_engine, text

from datasets import Dataset

import ragas
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    answer_correctness,
    answer_similarity
)
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

from dotenv import load_dotenv, find_dotenv
from typing import Dict, Any, Tuple, Optional
import torch
import os
import re
import json
import csv
# import evaluate


# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# Change the current working directory to the project root
os.chdir(project_root_path)


# --- FIX 2: Bypass find_dotenv() and use a direct, verified path ---
dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# Add a critical check to ensure the .env file exists at this path
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(
        f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
        f"Please double-check the path you pasted into 'project_root_path'."
    )


# Load the .env file from the explicit, verified path
load_dotenv(dotenv_path=dotenv_path)

# The project root is now simply the current working directory
project_root = os.getcwd()

# --- Now, the rest of your variable loading will work correctly ---
relative_data_dir = os.getenv("SQL_DATASET_DIR")

# Add a check to make sure the variable was loaded successfully from the file
if not relative_data_dir:
    raise ValueError(
        "ERROR: 'SQL_DATASET_DIR' was not found in your .env file, or the file is empty."
    )

data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- Final Verification ---
print(f"✅ Project root successfully set to: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

✅ Project root successfully set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
📁 Data directory set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/SQL_Dataset


## Helper Function to Sanitize File Names

In [2]:
def sanitize_table_name(filename):
    """
    Cleans a filename to create a safe, SQL-compliant table name.
    - Converts to lowercase
    - Replaces spaces and hyphens with underscores
    - Removes all other non-alphanumeric characters (except underscores)
    """
    # Remove the .csv extension
    name = os.path.splitext(filename)[0]
    # Convert to lowercase and replace spaces/hyphens
    name = name.lower().replace(' ', '_').replace('-', '_')
    # Remove any remaining invalid characters
    name = re.sub(r'[^a-z0-9_]', '', name)
    return name

In [3]:
# Create an in-memory SQLite database
# This database exists only as long as the script is running
engine = create_engine("sqlite:///:memory:")

# --- Dynamically load all CLEANED CSVs from the 'SQL_Dataset' directory ---
# This should point to the folder where your 'run_SQL_cleaning.py' script saved the files.
sql_data_directory = "SQL_Dataset" 
table_names = [] # To keep track of the tables we create

print(f"Searching for cleaned CSV files to ingest in '{sql_data_directory}'...")

# Check if the directory exists to avoid errors
if not os.path.isdir(sql_data_directory):
    print(f"Error: The directory '{sql_data_directory}' was not found. Please ensure the cleaning script ran successfully.")
else:
    for filename in os.listdir(sql_data_directory):
        if filename.endswith(".csv"):
            try:
                file_path = os.path.join(sql_data_directory, filename)
                
                # 1. Load the already-cleaned CSV into a DataFrame
                cleaned_df = pd.read_csv(file_path)
                
                # 2. Create a clean table name from the filename
                # Example: "cleaned_m891481.csv" -> "cleaned_m891481"
                table_name = sanitize_table_name(filename)
                table_names.append(table_name)
                
                # 3. Ingest the cleaned DataFrame into the SQL database
                cleaned_df.to_sql(table_name, engine, index=False, if_exists='replace')
                
                print(f" - Successfully ingested '{filename}' into SQL table '{table_name}'")
            except Exception as e:
                print(f" - FAILED to ingest {filename}. Error: {e}")

print(f"\nIn-memory SQL database created and populated with {len(table_names)} table(s).")
sql_database = SQLDatabase(engine)


Searching for cleaned CSV files to ingest in 'SQL_Dataset'...
 - Successfully ingested 'Convicted Penal Population by Age Group (2006-2020).csv' into SQL table 'convicted_penal_population_by_age_group_2006_2020'
 - Successfully ingested 'Convicted Penal Population by Age Group (2020 onwards).csv' into SQL table 'convicted_penal_population_by_age_group_2020_onwards'
 - Successfully ingested 'Convicted Penal Population by Age Group and Offence Group (2006-2020).csv' into SQL table 'convicted_penal_population_by_age_group_and_offence_group_2006_2020'
 - Successfully ingested 'Convicted Penal Population by Age Group and Offence Group (2020 onwards).csv' into SQL table 'convicted_penal_population_by_age_group_and_offence_group_2020_onwards'
 - Successfully ingested 'Convicted Penal Population by Education Level.csv' into SQL table 'convicted_penal_population_by_education_level'
 - Successfully ingested 'Convicted Penal Population by Gender and Offence Group.csv' into SQL table 'convicted_pe

## Llama 3.1 8B Instruct

In [4]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Initialize the tokenizer to get the token ID for our stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# The semicolon is our desired stop character. Get its token ID.
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Now, initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    model_kwargs={"token": hf_token, "torch_dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # This tells the model to stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.


## Generate Summary of each SQL Table

In [5]:
summary_prompt_str = """\
Provide a short, one-sentence summary for a table that has the following columns.
Your response MUST be ONLY the summary text and nothing else.

Columns:
{table_columns}

Summary: """
summary_prompt_tmpl = PromptTemplate(summary_prompt_str)

table_summaries = {}

print("--- Starting one-time summary generation ---")
print("This may take some time and consume a lot of memory.")

for table_name in table_names:
    try:
        print(f"  - Generating summary for: {table_name}")
        # Get ONLY the column names to save memory
        df = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 1", engine)
        table_columns = str(df.columns.tolist())

        # Generate the summary using your local LLM
        summary = llm.predict(summary_prompt_tmpl, table_columns=table_columns).strip()
        table_summaries[table_name] = summary
        print(f"    -> Success.")

    except Exception as e:
        print(f"    -> FAILED to generate summary for {table_name}. Assigning default. Error: {e}")
        table_summaries[table_name] = "No summary available for this table."

# Save the generated summaries to a file
summary_file_path = "table_summaries.json"
with open(summary_file_path, 'w') as f:
    json.dump(table_summaries, f, indent=4)

print(f"\n--- Summaries saved to '{summary_file_path}' ---")
print(json.dumps(table_summaries, indent=4))

Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


--- Starting one-time summary generation ---
This may take some time and consume a lot of memory.
  - Generating summary for: convicted_penal_population_by_age_group_2006_2020


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


    -> Success.
  - Generating summary for: convicted_penal_population_by_age_group_2020_onwards


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


...

    -> Success.
  - Generating summary for: convicted_penal_population_by_gender_and_offence_group


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


    -> Success.
  - Generating summary for: convicted_penal_population_by_gender


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


    -> Success.
  - Generating summary for: convicted_penal_population_by_offence_group
    -> Success.

--- Summaries saved to 'table_summaries.json' ---
{
    "convicted_penal_population_by_age_group_2006_2020": "The table displays the population of a given area by age group and the total number of population for each year.  The data is organized in a way that allows for easy comparison of population trends over time.  The table includes the year, the population by age group, and the total number of population.  The data is presented in a clear and concise manner, making it easy to analyze and understand.  The table is useful for researchers, policymakers, and other stakeholders who need to understand population dynamics and trends.  The table can be used to identify patterns and correlations between population growth and other factors such as economic development, education, and healthcare.  The table can also be used to inform policy decisions and resource allocation.  The table is

## Context for Text-To-SQL LLM

In [6]:
schema_parts = []
summary_file_path = "table_summaries.json"

print(f"--- Loading pre-computed summaries from '{summary_file_path}' ---")
with open(summary_file_path, 'r') as f:
    table_summaries = json.load(f)

# Build the context string using the loaded summaries
for table_name in table_names:
    summary = table_summaries.get(table_name, "No summary available.")
    raw_schema = sql_database.get_single_table_info(table_name)
    schema_parts.append(
        f"Table Name: {table_name}\n"
        f"Table Summary: {summary}\n"
        f"Table Schema: {raw_schema}"
    )

schema_info = "\n\n".join(schema_parts)
print("\n--- Final context being sent to Text-to-SQL LLM ---")
print(schema_info)

--- Loading pre-computed summaries from 'table_summaries.json' ---

--- Final context being sent to Text-to-SQL LLM ---
Table Name: convicted_penal_population_by_age_group_2006_2020
Table Summary: The table displays the population of a given area by age group and the total number of population for each year.  The data is organized in a way that allows for easy comparison of population trends over time.  The table includes the year, the population by age group, and the total number of population.  The data is presented in a clear and concise manner, making it easy to analyze and understand.  The table is useful for researchers, policymakers, and other stakeholders who need to understand population dynamics and trends.  The table can be used to identify patterns and correlations between population growth and other factors such as economic development, education, and healthcare.  The table can also be used to inform policy decisions and resource allocation.  The table is a valuable tool f

## Raw SQL Query

In [7]:
# Define user query
query_text_sql = "What was the total convicted penal population for the '21-30' age group in the year 2010?"

text_to_sql_prompt_template_str = (
        "You are an expert SQL generator. Analyze the user's question and the provided database context to generate a single, syntactically correct SQLite query.\n\n"
        "### INSTRUCTIONS\n"
        "1. Examine the table summaries to understand what data is in each table.\n"
        "2. IMPORTANT: If the user's question can be answered using a single table, you MUST use only that table. Do not create unnecessary JOINs.\n"
        "3. For string comparisons in WHERE clauses, use the `LOWER()` function on both the column and the value to ensure case-insensitive matching.\n"
        "4. Your response MUST be ONLY the single, raw SQL query.\n\n"
        "### DATABASE CONTEXT\n{schema}\n\n"
        "### QUESTION\n{query_str}\n\n"
        "### SQL QUERY\n"
    )
text_to_sql_prompt = PromptTemplate(text_to_sql_prompt_template_str)
    
# Generate the query using the main LLM
raw_sql_response = llm.predict(text_to_sql_prompt, schema=schema_info, query_str=query_text_sql)

Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


## Clean SQL Query

In [8]:
# Define the cleaning function
def extract_first_sql_query(raw_text: str) -> str:
    match = re.search(r"SELECT\s.*?;", raw_text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(0).strip() if match else ""
    
clean_sql_query = extract_first_sql_query(raw_sql_response)
print(f"\n--- Cleaned SQL to be executed ---\n{clean_sql_query}")


--- Cleaned SQL to be executed ---
SELECT number_of_population 
FROM convicted_penal_population_by_age_group_2006_2020 
WHERE LOWER(population_by_age_group) = '21-30' AND year = 2010;


## Synthesize SQL Prompt

In [9]:
# Cell: Synthesize SQL Prompt

response_from_db = sql_database.run_sql(clean_sql_query)

# --- DEBUG: Print the raw response from the database ---
print(f"DEBUG: Raw response from DB: {response_from_db}")
print(f"DEBUG: Type of DB response: {type(response_from_db)}")

def extract_scalar(sql_result):
    """
    Robustly extracts a single scalar value from various SQL result formats,
    including the tuple format from the LlamaIndex SQLDatabase wrapper.
    """
    if not sql_result:
        return None
    
    if isinstance(sql_result, tuple) and len(sql_result) > 0:
        data_to_parse = sql_result[0]
    else:
        data_to_parse = sql_result

    # Handle string representation of lists/tuples, e.g., "'[(2206,)]'"
    if isinstance(data_to_parse, str):
        try:
            # Safely evaluate the string into a Python object
            import ast
            data_to_parse = ast.literal_eval(data_to_parse)
        except (ValueError, SyntaxError):
            # If it's just a plain number string, return it
            if data_to_parse.strip().isdigit():
                return int(data_to_parse.strip())
            return None # Not a recognized format

    # Handle if the data is already in a list/tuple format
    if isinstance(data_to_parse, list) and len(data_to_parse) > 0:
        row = data_to_parse[0]
        if isinstance(row, (list, tuple)) and len(row) > 0:
            return row
        if isinstance(row, dict):
            # Return the first value from the dictionary
            return next(iter(row.values()), None)
    
    # Handle the case where the result is already a scalar
    if isinstance(sql_result, (int, float)):
        return sql_result
        
    return None

value = extract_scalar(response_from_db)

# --- DEBUG: Print the extracted value ---
print(f"DEBUG: Extracted value: {value}")

if value is None:
    clean_final_response = (
        "I couldn’t find a recorded value for the '21–30' age group in 2010 in the current dataset."
    )
else:
    synthesis_prompt_str = (
        "You are a helpful assistant. Based on the user's question and the data retrieved from the database, "
        "provide a clear, conversational, and complete single-sentence answer.\n\n"
        "### User's Question:\n{original_question}\n\n"
        "### Data from Database:\n{sql_result}\n\n"
        "### Answer:"
    )
    synthesis_prompt = PromptTemplate(synthesis_prompt_str)

    # Define multiple stop tokens to cleanly end the sentence
    # Stop at a period, a newline, or the model's official end-of-text token
    stop_tokens = [".", "\n", "<|end_of_text|>"]
    stop_token_ids = [tokenizer.convert_tokens_to_ids(token) for token in stop_tokens]
    stop_token_ids = [tid for tid in stop_token_ids if isinstance(tid, int)] # Ensure all are valid IDs
    original_eos_token_id = llm.generate_kwargs.get('eos_token_id')
    llm.generate_kwargs['eos_token_id'] = stop_token_ids
    
    final_response = llm.predict(
        synthesis_prompt,
        original_question=query_text_sql,
        sql_result=str(value) # Pass the clean value
    )
    clean_final_response = final_response.strip()


Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.


DEBUG: Raw response from DB: ('[(2206,)]', {'result': [(2206,)], 'col_keys': ['number_of_population']})
DEBUG: Type of DB response: <class 'tuple'>
DEBUG: Extracted value: (2206,)


## SQL Example Usage

In [10]:
print(f"--- Running query: \"{query_text_sql}\" ---")

try:
    print("\n" + "="*20 + " FINAL ANSWER " + "="*20)
    print(textwrap.fill(clean_final_response, 80))

except Exception as e:
    print(f"\nAn error occurred during the manual workflow: {e}")


--- Running query: "What was the total convicted penal population for the '21-30' age group in the year 2010?" ---

==================== FINAL ANSWER ====================
The total convicted penal population for the '21-30' age group in the year 2010
was 2206.


## Benchmarking

In [18]:
# --- Configuration ---
relative_benchmark_path = os.getenv("SQL_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("SQL_BENCHMARK_DATASET_DIR is not set in your .env file.")

BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "ragas_evaluation_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

judge_llm = llm_factory(model="gpt-4o")
judge_embeddings = embedding_factory(model="text-embedding-ada-002")

# --- Load Benchmark Data ---
rows = []
print(f"Loading questions from: {BENCHMARK_FILE_PATH}")
with open(BENCHMARK_FILE_PATH, mode='r', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = [row for row in reader]
# rows = rows[:3] # Uncomment for a quick test run
print(f"Loaded {len(rows)} questions to benchmark.")

# --- Define Your Pipeline's Prompts ---
text_to_sql_prompt = PromptTemplate(
    "You are an expert SQL generator. Analyze the user's question and the provided database context to generate a single, syntactically correct SQLite query. Your response MUST be ONLY the single, raw SQL query.\n\n"
    "### DATABASE CONTEXT\n{schema}\n\n"
    "### QUESTION\n{query_str}\n\n"
    "### SQL QUERY\n"
)
synthesis_prompt = PromptTemplate(
    "Based on the user's question and the SQL result, provide a conversational, single-sentence answer. If the result is empty or an error, state that the data could not be found.\n\n"
    "Question: {original_question}\n"
    "SQL Result: {sql_result}\n\n"
    "Answer:"
)
period_token_id = tokenizer.convert_tokens_to_ids(".")

# --- Run Pipeline and Collect Data for Ragas ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
print("\n--- Running pipeline for each benchmark question ---")
for i, row in enumerate(rows, 1):
    q = row.get("Question", row.get("question", "")).strip()
    gold_answer = row.get("Answer", row.get("answer", "")).strip()
    
    # --- FIX: Initialize context as an empty list for consistency ---
    retrieved_contexts = []
    predicted_answer = "ERROR: Pipeline failed."

    try:
        raw_sql = llm.predict(text_to_sql_prompt, schema=schema_info, query_str=q)
        generated_sql = extract_first_sql_query(raw_sql)
        
        sql_result_str = ""
        if generated_sql:
            try:
                sql_result = sql_database.run_sql(generated_sql)
                sql_result_str = str(sql_result)
                tables_ref = [t for t in table_names if t in generated_sql]
                if tables_ref:
                    # This now correctly assigns a list of strings
                    retrieved_contexts = [sql_database.get_single_table_info(t) for t in tables_ref]
            except Exception as e:
                sql_result_str = f"ERROR executing SQL: {e}"
        else:
            sql_result_str = "ERROR: No SQL query was generated."

        gen = llm.predict(
            synthesis_prompt,
            original_question=q,
            sql_result=sql_result_str,
            eos_token_id=period_token_id,
        )
        predicted_answer = gen.strip()

    except Exception as e:
        predicted_answer = f"ERROR: A critical pipeline error occurred: {e}"

    ragas_data["question"].append(q)
    ragas_data["answer"].append(predicted_answer)
    ragas_data["contexts"].append(retrieved_contexts)
    ragas_data["ground_truth"].append(gold_answer)
    
    print(f"Processed {i}/{len(rows)}...")

# Run Ragas Evaluation
print("\n--- Preparing data and running Ragas evaluation ---")
ragas_dataset = Dataset.from_dict(ragas_data)

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    answer_correctness,
]

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("✅ Ragas evaluation complete.")

# Display and Save Results
df_results = result.to_pandas()
df_results.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed Ragas evaluation results saved to: {OUTPUT_FILE_PATH}")

print("\n--- Overall Ragas Performance Metrics ---")
print(df_results.mean().to_string())
print("-----------------------------------------")

Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.


Loading questions from: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Benchmark Dataset/sql_benchmark.csv
Loaded 100 questions to benchmark.

--- Running pipeline for each benchmark question ---


Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.


Processed 1/100...


Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.


Processed 100/100...

--- Preparing data and running Ragas evaluation ---


Evaluating:   0%|          | 0/300 [00:00<?, ?it/s]

✅ Ragas evaluation complete.

Detailed Ragas evaluation results saved to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/ragas_evaluation_results.csv

--- Overall Ragas Performance Metrics ---
answer_relevancy      0.887413
faithfulness          0.005000
answer_correctness    0.737181
-----------------------------------------


/tmp/ipykernel_13927/1665438853.py:109: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  print(df_results.mean().to_string())
